# Business Forecasting · Lecture 2 — Understanding and Decomposing Time Series
### A fully explained, executable Python / Google Colab student reference

**Use only the original data supplied for this lecture.** No part of this notebook downloads data or creates replacement time series. It works with the four original CSVs from the provided book-data ZIP:

| CSV file | What the book dataset measures | What we use it for |
|---|---|---|
| `global_economy.csv` | Annual indicators for countries/regions; includes Australian GDP, population, CPI and exports | Per-capita adjustment, inflation index and centered moving averages |
| `aus_retail.csv` | Monthly turnover by Australian state and retail industry | Calendar-day adjustment, inflation-adjusted turnover and transformations |
| `us_employment.csv` | Monthly employment by US industry, with values in **thousands of persons** | STL, seasonal adjustment, 2×12-MA and classical decomposition |
| `aus_production.csv` | Quarterly Australian production indicators | Beer production and the 2×4 moving average |

**How to study this notebook:** Read each explanation, run the code cell immediately below it, read the comments beside/above the code lines, and then interpret the printed result or graph. Run cells **from top to bottom**; later cells reuse datasets and variables created in earlier cells.

**What you will learn:** (1) calendar, population and inflation adjustments; (2) transformations; (3) observed/trend/seasonal/remainder; (4) STL and seasonal adjustment; (5) moving averages, odd/even centering and endpoint loss; (6) classical decomposition; and (7) a reusable starter recipe for independent Python work.

**Important:** The official files are a fixed, historical teaching dataset. We analyze the time spans actually present in those files; this notebook makes no claim that the figures are current.

## 0 · How Colab, code cells and comments work

- A **code cell** runs Python; press **Shift + Enter** or the Play button. An **explanation cell** (like this one) does not run code.
- A line that begins with `#` is a **comment**: Python does not execute it. It explains the purpose of the code that follows.
- `df` usually means **DataFrame**: a table of rows and columns. A **Series** is one column of a DataFrame.
- `pd`, `np` and `plt` are conventional short names, called *aliases*, for `pandas`, `numpy` and `matplotlib.pyplot`.
- A **time series** is a variable observed in chronological order, such as monthly employment. Always check that its date column is interpreted correctly and that its rows are sorted.

**Data-upload requirement:** At the upload step you may select either **one ZIP** (recommended) or the **four individual CSVs**. If you select only some CSVs, the notebook stops and tells you precisely which are missing. The ZIP may contain a nested folder; the notebook finds the required filenames inside it. No Google Drive connection or internet data source is needed.

## 1 · Core Python packages — remember this setup

We begin with imports. Importing does **not** open a data file or draw a graph: it makes specific tools available for the remaining cells.

- `Path` gives predictable file/folder paths.
- `zipfile` and `io` let Python open an uploaded ZIP **in memory** and save the needed CSVs.
- The Colab upload tool opens a file picker in Colab; in ordinary Jupyter, a safe import fallback uses existing local files.
- `numpy` provides numerical transformations, `pandas` provides tables and date handling, and `matplotlib` draws graphs.
- `STL` and `seasonal_decompose` provide two approaches to time-series decomposition. `STL` estimates a time-varying seasonal component, while *classical decomposition* estimates one repeating seasonal profile.
- `display` shows tables neatly inside Colab.

All tools used for the analysis are common Colab packages. This notebook does **not** install anything or download data.

In [ ]:
# Import Path to create file and folder paths on any supported operating system.
from pathlib import Path

# Import zipfile to read a ZIP archive uploaded by a student.
import zipfile

# Import io to treat the uploaded ZIP bytes like a file without downloading anything.
import io

# Import NumPy as np for numerical operations (for example, sqrt and log).
import numpy as np

# Import pandas as pd for CSV files, tables, dates, grouping and moving averages.
import pandas as pd

# Import pyplot as plt for publication-style graphs.
import matplotlib.pyplot as plt

# Import STL for decomposition into a flexible trend, seasonality and remainder.
from statsmodels.tsa.seasonal import STL

# Import seasonal_decompose for the classical additive and multiplicative methods.
from statsmodels.tsa.seasonal import seasonal_decompose

# Import display so we can print pandas tables clearly below the code cells.
from IPython.display import display

# Make new plots wide enough for monthly and annual dates to be readable.
plt.rcParams['figure.figsize'] = (11, 5)

# Show light horizontal and vertical grid lines to help read dates and values.
plt.rcParams['axes.grid'] = True

# List the four exact filenames students must supply from the book-data package.
REQUIRED_FILES = ['global_economy.csv', 'aus_retail.csv', 'us_employment.csv', 'aus_production.csv']

# Create a dedicated folder so future cells always read files from one known location.
DATA_DIR = Path('Lecture2_Data')

# Create the folder if it does not exist; do nothing if it already exists.
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Confirm that imports and folder preparation worked.
print('Python setup ready. Data folder:', DATA_DIR.resolve())

## 2 · Upload the original dataset (ZIP **or** individual CSVs)

**What happens here?** Colab provides the selected file bytes. If you upload a ZIP, the code opens it, searches all subfolders for the **four exact CSV filenames**, and copies only those four into `Lecture2_Data`. If you upload CSVs individually, it saves them in the same folder. Extra files in the ZIP are ignored.

**First time:** Click the file picker and choose `Lecture2_OriginalBookData_CSVs.zip` **or** choose all four CSVs together. The larger original all-data ZIP also works, because the required files are found by filename even inside its `data/` folder. **No data are downloaded from the web.**

**Repeat run:** If the four files are still in Colab's temporary storage, cancel the picker or rerun the notebook and select them again. Colab's files are temporary: a disconnected runtime may need a new upload.

**Outside Google Colab:** The uploader is skipped; put the four files inside `Lecture2_Data/` in the notebook's working directory. This makes the same notebook testable in ordinary Jupyter.

In [ ]:
# Try to import the Colab uploader; ordinary Jupyter does not provide this package.
try:
    # Import Google's Colab upload interface when it is actually installed.
    from google.colab import files
except ImportError:
    # Outside Colab, read locally present CSVs from the known data folder instead.
    uploaded_files = {}
else:
    # In Colab, open the file picker and receive filename -> uploaded bytes.
    uploaded_files = files.upload()

# Process each file that the student selected in the upload dialog.
for uploaded_name, uploaded_bytes in uploaded_files.items():
    # Normalize the uploaded name so the extension check is case-insensitive.
    upload_extension = Path(uploaded_name).suffix.lower()

    # Branch A: a single ZIP can contain nested folders and many unnecessary files.
    if upload_extension == '.zip':
        # Open the ZIP directly from the uploaded bytes without using the internet.
        with zipfile.ZipFile(io.BytesIO(uploaded_bytes)) as archive:
            # Examine every entry (file or folder) stored inside this ZIP.
            for item in archive.infolist():
                # Take only the filename and ignore parent folder paths in the archive.
                clean_name = Path(item.filename).name

                # Extract only a required CSV, never unrelated or unsafe ZIP entries.
                if not item.is_dir() and clean_name in REQUIRED_FILES:
                    # Read the CSV bytes from inside the ZIP into the known data folder.
                    (DATA_DIR / clean_name).write_bytes(archive.read(item))

                    # Tell the student exactly which required file was obtained.
                    print('Extracted from ZIP:', clean_name)

    # Branch B: save any separately uploaded CSV that is part of our required list.
    elif uploaded_name in REQUIRED_FILES:
        # Write this required CSV to the same data folder as ZIP-extracted files.
        (DATA_DIR / uploaded_name).write_bytes(uploaded_bytes)

        # Confirm the uploaded CSV has been saved successfully.
        print('Saved uploaded CSV:', uploaded_name)

    # Branch C: other uploaded filenames are not needed for this lecture.
    else:
        # Tell the student why an extra file is being ignored.
        print('Ignored unrelated file:', uploaded_name)

# List every required file not yet present in the data folder.
missing_files = [name for name in REQUIRED_FILES if not (DATA_DIR / name).is_file()]

# Stop before analysis if students did not provide the complete dataset.
if missing_files:
    # Raise a helpful error with all missing filenames rather than failing later.
    raise FileNotFoundError('Please upload the ZIP or these missing CSVs: ' + ', '.join(missing_files))

# Confirm that the entire lecture can now use local files only.
print('All four required book CSVs are available locally. Ready to analyze!')

## 3 · Open the book CSV files and verify the data structure

**Theory before coding.** A forecasting workflow begins with **data understanding**, not model fitting. We need to identify the observation frequency (annual, quarterly, monthly), what one row represents, and whether a series is identified by a country or industry. A date can look correct as text but behave incorrectly in graphs and sorting, so we deliberately convert monthly/quarterly dates to `datetime`.

**Actual schema of these exact files:**

| Variable | Meaning |
|---|---|
| `unique_id` in `global_economy.csv` | Country/region, such as `Australia` |
| `ds` in `global_economy.csv` | Calendar **year**, expressed as an integer |
| `GDP`, `Population`, `CPI`, `Exports` | Annual total GDP, population, price index, and exports as a share of GDP |
| `State`, `Industry`, `Month`, `Turnover` | One Australian state, industry, month and its turnover |
| `unique_id`, `ds`, `y` in `us_employment.csv` | Employment industry, monthly date and employed persons **in thousands** |
| `ds`, `Beer` in `aus_production.csv` | Quarter start date and beer production |

**Warning about grouping:** The Australian retail file has separate rows for states. Plotting one state's food retailing and calling it “Australia” is incorrect. For national examples, we **sum states for each calendar month**. The supplied file has fewer early observations for the Northern Territory; the national total uses the states represented in the original file for each month.

In [ ]:
# Read the annual economy CSV into a pandas DataFrame (one row per country and year).
global_economy = pd.read_csv(DATA_DIR / 'global_economy.csv')

# Read the retail CSV; parse_dates converts Month from text to a real datetime value.
aus_retail = pd.read_csv(DATA_DIR / 'aus_retail.csv', parse_dates=['Month'])

# Read monthly employment; parse ds as datetime so filtering and plotting work correctly.
us_employment = pd.read_csv(DATA_DIR / 'us_employment.csv', parse_dates=['ds'])

# Read quarterly production and interpret quarter-start dates as datetime values.
aus_production = pd.read_csv(DATA_DIR / 'aus_production.csv', parse_dates=['ds'])

# Show each file's rows and columns; shape returns (number of rows, number of columns).
for name, table in [('Economy', global_economy), ('Retail', aus_retail),
                    ('Employment', us_employment), ('Production', aus_production)]:
    # Print the file label, shape and real column names so students avoid spelling mistakes.
    print(f'{name}: shape={table.shape}, columns={table.columns.tolist()}')

# Display actual observations; head(3) means “show the first three rows.”
display(global_economy.head(3))

# Display an example of monthly retail dates and turnover values.
display(aus_retail.head(3))

# Display monthly employment fields: industry, date and y (thousands of people).
display(us_employment.head(3))

# Display quarterly Australian production and the Beer column.
display(aus_production[['ds', 'Beer']].head(3))

### 3.1 · Verify the files before trusting a graph

A graph drawn from the wrong subgroup can look entirely reasonable and still be wrong. We check the exact target labels, sort dates, inspect missing values, and verify that positive-value methods will be mathematically legal. We do **not** silently fill missing observations with invented numbers.

In [ ]:
# Show the actual annual years covered by the global economy file.
print('Economy years:', global_economy['ds'].min(), 'to', global_economy['ds'].max())

# Show the actual first and last months present in the retail file.
print('Retail months:', aus_retail['Month'].min().date(), 'to', aus_retail['Month'].max().date())

# Show the actual first and last months present in the US employment file.
print('Employment months:', us_employment['ds'].min().date(), 'to', us_employment['ds'].max().date())

# Show the actual first and last quarterly observations in the production file.
print('Production quarters:', aus_production['ds'].min().date(), 'to', aus_production['ds'].max().date())

# Count missing values in each economy field: an empty value can break selected methods.
print('Missing economy values by column:')

# Show the count of absent entries for each global-economy column.
print(global_economy.isna().sum().to_string())

# Require Australia to be an actual country label in the supplied economy file.
assert (global_economy['unique_id'] == 'Australia').any(), 'Australia not found in global_economy.csv.'

# Require the Retail Trade label to exist in the supplied employment file.
assert (us_employment['unique_id'] == 'Retail Trade').any(), 'Retail Trade not found in us_employment.csv.'

# Require the Food retailing industry, used by our transformation example.
assert (aus_retail['Industry'] == 'Food retailing').any(), 'Food retailing not found in aus_retail.csv.'

# Require the print-media industry, used by our inflation example.
assert (aus_retail['Industry'] == 'Newspaper and book retailing').any(), 'Newspaper and book retailing not found.'

# Verify that the production dataset really has the Beer variable.
assert 'Beer' in aus_production.columns, 'Beer column not found in aus_production.csv.'

# Confirm that the data labels needed throughout this tutorial exist.
print('Dataset checks passed: lecture examples match the supplied files.')

---
# Part A · Historical data adjustments

**Why adjust historical observations?** A time series may move for a reason that has nothing to do with the business question we want to answer. February has fewer calendar days than March; GDP changes with the number of people; and money loses or gains purchasing power as prices move. An adjustment tries to make observations **more comparable**, but it does not automatically make a forecast accurate. A good adjustment must match the decision and unit of analysis.

## 4 · Calendar-day adjustment — Australian food retailing

**Dataset:** `aus_retail.csv` contains monthly turnover for many Australian states and industries. Select `Food retailing`, then sum all available state observations for each month to make a national series. `Turnover` is a monthly sales measure, not “sales per day.”

**Problem:** Even if stores sold exactly the same amount per day, a 31-day month would have a larger monthly total than a 28-day month. **Calendar-day adjustment** calculates `monthly turnover / calendar days in that month` and changes the unit to average turnover per calendar day.

**Distinction:** Calendar days are not necessarily **trading days**. To calculate sales per trading day, we would need a reliable count of each month's trading days and any relevant closures. That information is **not in these four CSVs**; do not present calendar-day turnover as a trading-day adjustment.

**Observe:** The two graphs have different units. Compare the size of month-to-month variation, not their y-axis numeric scales. Average turnover/day may still be seasonal because demand really can change across the year.

In [ ]:
# Choose the Food retailing industry; copy keeps the analysis separate from the full file.
food_rows = aus_retail.loc[aus_retail['Industry'] == 'Food retailing'].copy()

# Sum available state turnover within each month to get one national monthly observation.
food_monthly = food_rows.groupby('Month', as_index=False)['Turnover'].sum()

# Ensure the new national series is strictly in chronological order.
food_monthly = food_monthly.sort_values('Month').reset_index(drop=True)

# Use pandas' date accessor to count actual calendar days (28, 29, 30 or 31).
food_monthly['days_in_month'] = food_monthly['Month'].dt.days_in_month

# Divide the monthly total by days, changing the unit to turnover per calendar day.
food_monthly['turnover_per_calendar_day'] = food_monthly['Turnover'] / food_monthly['days_in_month']

# Display values so students can check one division by hand before trusting a graph.
display(food_monthly[['Month', 'Turnover', 'days_in_month', 'turnover_per_calendar_day']].head(6))

# Show how many different state records contribute to each month's total.
state_counts = food_rows.groupby('Month')['State'].nunique()

# Print the observed minimum and maximum state counts; do not invent missing state data.
print('Number of states represented per month: minimum', state_counts.min(), 'maximum', state_counts.max())

In [ ]:
# Start a new figure for the ORIGINAL monthly turnover series.
plt.figure()

# Draw the unadjusted national monthly total in chronological order.
plt.plot(food_monthly['Month'], food_monthly['Turnover'])

# Name what is measured rather than just writing “sales.”
plt.title('Australian food retailing — monthly turnover')

# Label the horizontal axis with its time unit.
plt.xlabel('Month')

# Label the vertical axis with the original total rather than a daily average.
plt.ylabel('Monthly turnover (original data units)')

# Keep titles and labels from being cut off.
plt.tight_layout()

# Display this figure inside Colab.
plt.show()

# Start a separate figure so the different measurement scale is unambiguous.
plt.figure()

# Draw daily-average turnover obtained from the same monthly observations.
plt.plot(food_monthly['Month'], food_monthly['turnover_per_calendar_day'])

# Label the adjusted quantity correctly.
plt.title('Australian food retailing — turnover per calendar day')

# Show that the horizontal axis still represents months.
plt.xlabel('Month')

# Distinguish a daily average from monthly totals.
plt.ylabel('Average turnover per calendar day')

# Fit the plot elements inside the figure.
plt.tight_layout()

# Show the second figure.
plt.show()

**Interpretation checkpoint:** For a specific row, divide `Turnover` by `days_in_month` yourself. If February's monthly total is below March's but the daily average is similar, month length explains part of the raw difference. Do **not** conclude that dividing by days removes holiday demand, trend or all seasonality; it removes only the arithmetic effect of different calendar-month lengths.

## 5 · Population adjustment — Australian GDP per capita

**Dataset:** In `global_economy.csv`, `unique_id` identifies each country and `ds` is the **year**. Keep Australia only. `GDP` is a total economic output measure and `Population` is the number of people. GDP per capita is `GDP / Population`.

**Why both plots?** Total GDP can increase partly because the population increases. Per-capita GDP asks a different question: how much GDP is produced **per person** on average. An increase in total GDP does not guarantee an equal increase per person.

**Important limitations:** The supplied GDP is a monetary value in the original book file, not a measure we have converted to constant prices. GDP per capita computed here is **not** a price-adjusted living-standard index. It is also not the income of a typical individual.

In [ ]:
# Filter the multi-country table to rows labeled Australia.
australia = global_economy.loc[global_economy['unique_id'] == 'Australia'].copy()

# Sort annual observations before calculating or plotting across time.
australia = australia.sort_values('ds').reset_index(drop=True)

# Prevent invalid division if the population were missing or zero.
assert australia['Population'].notna().all() and (australia['Population'] > 0).all()

# Divide annual total GDP by the corresponding annual population value.
australia['GDP_per_capita'] = australia['GDP'] / australia['Population']

# Show a few actual years and the calculated per-person quantity.
display(australia[['ds', 'GDP', 'Population', 'GDP_per_capita']].head(6))

# Start a graph of total GDP; its vertical-axis units are NOT per person.
plt.figure()

# Draw GDP over the annual observations.
plt.plot(australia['ds'], australia['GDP'])

# Set a clear title naming the country and original total measure.
plt.title('Australia — total GDP')

# Label the time axis.
plt.xlabel('Year')

# Label the monetary measure without falsely calling it constant-price GDP.
plt.ylabel('Total GDP (dataset currency units)')

# Keep labels visible and show the plot.
plt.tight_layout()
plt.show()

# Open another figure because GDP/person has a different scale and business meaning.
plt.figure()

# Plot GDP per capita from the same years.
plt.plot(australia['ds'], australia['GDP_per_capita'])

# Name the adjusted measure in the figure title.
plt.title('Australia — GDP per capita')

# Show that each observation is annual.
plt.xlabel('Year')

# Specify that the data are now divided by population.
plt.ylabel('GDP per person (dataset currency units)')

# Display the graph without cropping labels.
plt.tight_layout()
plt.show()

## 6 · Inflation adjustment — Australian newspaper and book retailing

**Dataset combination:** The `Newspaper and book retailing` rows in `aus_retail.csv` provide monthly turnover by state. We add available states by month and then sum the 12 months into an **annual** turnover total. `global_economy.csv`, filtered to Australia, provides an **annual CPI** (Consumer Price Index) for the matching years.

**Why a merge?** We cannot divide a monthly value by an unrelated year's index. We create a common `year` variable and merge by that year. The price index has `CPI = 100` in **2010**, the reference year used in this supplied dataset.

**Formula:** `turnover_in_2010_dollars = observed_turnover × CPI_2010 / CPI_in_observation_year`.

- For years with CPI below 100, converting a nominal amount into 2010 purchasing-power units increases its numerical value.
- For years with CPI above 100, the adjusted number tends to be lower than the same nominal number.
- Price adjustment answers “what did turnover represent in *2010 purchasing-power units*?”, not “what was the number of books sold?”
- **Annual retail coverage caution:** Because one state's earliest rows may be absent, aggregating the source cannot recreate values that the supplied CSV does not include. We label the result as the turnover of the observed states in the dataset.

In [ ]:
# Keep only Australian newspaper and book retailing observations.
print_rows = aus_retail.loc[aus_retail['Industry'] == 'Newspaper and book retailing'].copy()

# Create the matching year from the monthly date; 2010-01 through 2010-12 become 2010.
print_rows['year'] = print_rows['Month'].dt.year

# Add reported turnover across states and all months inside each year.
print_annual = print_rows.groupby('year', as_index=False)['Turnover'].sum()

# Keep the same Australian economy rows already filtered in the previous section.
aus_cpi = australia[['ds', 'CPI']].copy()

# Rename the annual economy date column so both tables share “year.”
aus_cpi = aus_cpi.rename(columns={'ds': 'year'})

# Merge retail turnover with CPI for that SAME calendar year, keeping only matched years.
inflation_df = print_annual.merge(aus_cpi, on='year', how='inner', validate='one_to_one')

# Find the CPI specifically for the reference year 2010 in the original book file.
base_cpi = float(aus_cpi.loc[aus_cpi['year'] == 2010, 'CPI'].iloc[0])

# Verify that every matched CPI is positive so the adjustment can divide by it.
assert (inflation_df['CPI'] > 0).all(), 'Inflation adjustment requires positive CPI.'

# Put nominal turnover into the purchasing-power units of the 2010 reference year.
inflation_df['turnover_2010_prices'] = inflation_df['Turnover'] * base_cpi / inflation_df['CPI']

# Show the actual CPI base rather than assuming every downloaded CPI series uses 100.
print('CPI in 2010:', base_cpi)

# Display the inputs and adjusted output for several real matched years.
display(inflation_df[['year', 'Turnover', 'CPI', 'turnover_2010_prices']].head(8))

# Assert that the adjusted and nominal values match in the base year (CPI/CPI = 1).
assert np.isclose(float(inflation_df.loc[inflation_df['year'] == 2010, 'Turnover'].iloc[0]),
                  float(inflation_df.loc[inflation_df['year'] == 2010, 'turnover_2010_prices'].iloc[0]))

In [ ]:
# Open a single graph to compare two annual measures with the same turnover unit.
plt.figure()

# Plot nominal observed turnover in the currency values of each historical year.
plt.plot(inflation_df['year'], inflation_df['Turnover'], label='Nominal turnover')

# Plot inflation-adjusted turnover in the equivalent purchasing power of 2010.
plt.plot(inflation_df['year'], inflation_df['turnover_2010_prices'], label='Turnover in 2010 prices')

# Clearly identify the industry and meaning of the comparison.
plt.title('Australian newspaper and book retailing — inflation adjustment')

# Label annual observations.
plt.xlabel('Year')

# State that we are plotting annual turnover, not number of items sold.
plt.ylabel('Annual turnover (original data units)')

# Show which line is nominal and which uses 2010 prices.
plt.legend()

# Prevent plot labels from overlapping the image border.
plt.tight_layout()

# Draw the comparison chart.
plt.show()

**Read the result:** The blue/orange line color may vary by software settings; use the legend, not color memory. If nominal turnover holds steady while inflation-adjusted turnover declines, the same amount of currency represents less purchasing power. This is a **deflation of a nominal money series**, not a population adjustment. The two adjustments answer different business questions.

---
# Part B · Transformations: changing the scale to clarify patterns

**What is a transformation?** It creates a new series from the original observations. We may want the size of the seasonal ups-and-downs to be more stable as the series grows. Transformations do not remove a seasonal pattern by themselves: they change how the same pattern looks on a different numerical scale.

**Dataset:** National food retail turnover, formed from the original `Food retailing` rows in `aus_retail.csv`. We reuse the `food_monthly` table from the calendar-adjustment section so dates and values remain consistent throughout the lesson.

**Conditions:** Logarithms and reciprocal transformations require strictly **positive** input for these examples. Taking `np.log(0)` or dividing by zero is invalid. No dummy values or random noise are added to avoid that issue.

## 7 · Look at the original food-turnover series first

**What to see:** In an expanding retail series, absolute differences between peaks and troughs often become larger as the overall level increases. This motivates transformation—but do not decide from a single peak that every year is identical. First view the whole timeline.

In [ ]:
# Copy the correctly grouped national monthly food turnover data.
food = food_monthly[['Month', 'Turnover']].copy()

# Check positivity before logarithm and reciprocal calculations.
assert (food['Turnover'] > 0).all(), 'The transformations below need positive turnover values.'

# Plot the untransformed observations as the visual starting point.
plt.figure()

# Put month on the horizontal axis and original turnover on the vertical axis.
plt.plot(food['Month'], food['Turnover'])

# Identify this as the original, not an adjusted or transformed series.
plt.title('Australian food retailing — original monthly turnover')

# Label both axes in terms students can understand.
plt.xlabel('Month')
plt.ylabel('Turnover (original data units)')

# Display a clean plot.
plt.tight_layout()
plt.show()

## 8 · Square root, cube root, natural log and negative inverse

These transformations **compress** larger observations by different amounts. A simple way to compare them is to look at early and late seasonal swings **within each transformed plot**, not compare numerical values across different y-axis units.

| Transformation | Python | Meaning |
|---|---|---|
| Square root | `np.sqrt(y)` | Mild-to-moderate compression; values must be ≥ 0 |
| Cube root | `np.cbrt(y)` | A further compression for this positive series |
| Natural log | `np.log(y)` | Changes multiplicative/percentage-scale behavior to additive differences on a log scale |
| Negative inverse | `-1 / y` | Strong compression, with values approaching zero from below as original values grow |

**Why use a log?** A one-unit rise on the natural-log scale corresponds to multiplying the original positive observation by `e`. For **small** changes, a log difference of `0.01` is approximately a **1%** increase on the original scale; the exact percentage is `(exp(0.01) − 1) × 100`.

**No automatic winner:** Select a transformation based on the series' variability, interpretability and forecasting/model assumptions. More compression is not always better.

In [ ]:
# Take the square root of each positive monthly turnover observation.
food['square_root'] = np.sqrt(food['Turnover'])

# Take the real cube root of the same original observations, not of the square roots.
food['cube_root'] = np.cbrt(food['Turnover'])

# Take the natural log of the original observations; np.log is log base e.
food['log_turnover'] = np.log(food['Turnover'])

# Divide negative one by the original observations for a strong inverse transform.
food['negative_inverse'] = -1.0 / food['Turnover']

# Display the different scales side by side in a small sample of the same months.
display(food[['Month', 'Turnover', 'square_root', 'cube_root', 'log_turnover', 'negative_inverse']].head(6))

In [ ]:
# Create a four-panel figure so each different transformed scale gets its own y-axis.
fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# Draw square-root turnover in the upper-left panel.
axes[0, 0].plot(food['Month'], food['square_root'])

# State the transformation used in the upper-left panel.
axes[0, 0].set_title('Square root of turnover')

# Draw cube-root turnover in the upper-right panel.
axes[0, 1].plot(food['Month'], food['cube_root'])

# State the transformation used in the upper-right panel.
axes[0, 1].set_title('Cube root of turnover')

# Draw log turnover in the lower-left panel.
axes[1, 0].plot(food['Month'], food['log_turnover'])

# Identify the natural-log scale.
axes[1, 0].set_title('Natural log of turnover')

# Draw the negative inverse in the lower-right panel.
axes[1, 1].plot(food['Month'], food['negative_inverse'])

# Name the strongest transformation in the final panel.
axes[1, 1].set_title('Negative inverse of turnover')

# Loop over the four axes to label the same horizontal date unit.
for ax in axes.flat:
    # Put the month label on every panel so the axes are self-contained.
    ax.set_xlabel('Month')

    # Label the panel values as transformed values, not original currency turnover.
    ax.set_ylabel('Transformed value')

# Stop panel labels from running into adjacent plots.
plt.tight_layout()

# Display the four plots together.
plt.show()

### 8.1 · A numerical way to understand log changes

A difference in log values is **not** itself the percentage in exact arithmetic. We can transform a log difference back to its exact implied percentage: `(exp(change_in_log) − 1) × 100`. This matters when explaining results to managers.

In [ ]:
# Compute the one-month difference between successive log turnover values.
food['log_change'] = food['log_turnover'].diff()

# Convert the log change into its EXACT implied percentage change.
food['exact_percent_change'] = (np.exp(food['log_change']) - 1) * 100

# Calculate the SAME percentage change directly from original turnover as a cross-check.
food['direct_percent_change'] = food['Turnover'].pct_change() * 100

# Show the first few months; the very first has no earlier month for comparison.
display(food[['Month', 'log_change', 'exact_percent_change', 'direct_percent_change']].head(6))

# Verify both approaches give the same numerical percentage wherever defined.
assert np.allclose(food['exact_percent_change'].dropna(), food['direct_percent_change'].dropna())

# Confirm that the back-transformation calculation is correct.
print('Cross-check passed: exp(log difference) reproduces the direct percentage change.')

---
# Part C · Time-series patterns and STL decomposition

Before looking at a decomposition algorithm, identify what is visible on the **original time plot**:

- **Trend:** the sustained longer-term direction, which may change during the sample.
- **Seasonality:** repeated movements linked to a fixed, known calendar frequency; for monthly data with a yearly pattern the seasonal period is **12**.
- **Cycle:** rises and falls that do not recur at a fixed calendar period (for example, business expansions/recessions).
- **Remainder:** the portion that is not described by the estimated trend and seasonal components. The remainder can still contain meaningful structure; calling it “random” does not establish that it is independent noise.

### Decomposition models

**Additive:** `observed_t = trend_t + seasonal_t + remainder_t`. Seasonal effects are in the same units as the original variable; a `+100` employment seasonal effect means 100 thousand more persons relative to the estimated trend at that date.

**Multiplicative:** `observed_t = trend_t × seasonal_t × remainder_t`. Seasonal effects are factors: `1.02` means about 2% above the trend level; `0.98` means about 2% below.

**Rule of thumb:** If absolute seasonal swings are roughly constant as the series level changes, an additive model can be interpretable. If swings grow proportionally with the level, multiplicative treatment or a log transformation may be useful. These are descriptive choices, not proof that a specific forecasting model is correct.

## 9 · Select and inspect US retail employment

**Dataset:** `us_employment.csv` includes many industries, not only retail. `unique_id` identifies a series; `ds` is the month; `y` is the recorded number of employed people **in thousands**. Filter `unique_id == 'Retail Trade'` and choose observations starting in **January 1990**, matching this lecture's graph range.

**What you should see:** A large monthly time series with slower changes over time and repeated within-year differences. We will try to separate its observed movement into a trend-cycle line, a monthly seasonal contribution, and an unexplained remainder.

**Do not assume a time-series frequency from filename alone.** The code checks that each selected date advances one month. `asfreq('MS')` would insert missing calendar months if present; here we first test instead of silently imputing observations.

In [ ]:
# Create a mask selecting only the Retail Trade series and dates from January 1990 onward.
retail_mask = (us_employment['unique_id'] == 'Retail Trade') & (us_employment['ds'] >= '1990-01-01')

# Copy these rows so the original multi-industry DataFrame remains unchanged.
employment = us_employment.loc[retail_mask, ['ds', 'unique_id', 'y']].copy()

# Sort ascending by month, which is required for time-series calculations.
employment = employment.sort_values('ds').reset_index(drop=True)

# Make sure duplicate dates are not silently counted twice.
assert not employment['ds'].duplicated().any(), 'Duplicate months in the selected Retail Trade series.'

# Create the complete expected sequence of monthly first-of-month dates.
expected_months = pd.date_range(start=employment['ds'].min(), end=employment['ds'].max(), freq='MS')

# Verify that actual dates exactly match every expected month in this range.
assert employment['ds'].tolist() == list(expected_months), 'Missing or unexpected monthly dates in Retail Trade.'

# Confirm that there are no missing employment values to fill or estimate.
assert employment['y'].notna().all(), 'Missing employment observations require an explicit decision.'

# Print the exact sample size and endpoints supplied by the original file.
print('Retail Trade months:', len(employment))

# Show earliest and latest selected dates.
print('Sample dates:', employment['ds'].min().date(), 'to', employment['ds'].max().date())

# Display genuine observations and employment units from the source file.
display(employment.head(6))

# Start a figure showing the ORIGINAL, undecomposed series.
plt.figure()

# Plot monthly employment in thousands of people.
plt.plot(employment['ds'], employment['y'])

# Make the industry and series clear in the title.
plt.title('US Retail Trade employment — original monthly series')

# Label the horizontal time unit.
plt.xlabel('Month')

# Identify the magnitude: y is thousands of persons, NOT single persons.
plt.ylabel('Employed persons (thousands)')

# Show the plot without cutting labels.
plt.tight_layout()
plt.show()

## 10 · STL — Seasonal and Trend decomposition using LOESS

**STL** is a smoothing-based procedure that estimates the observed series as **trend + seasonal + remainder**. LOESS means *locally estimated scatterplot smoothing*; it fits smooth curves using nearby points rather than requiring one straight line across decades.

**What the model parameters mean:**

- `employment['y']`: the actual monthly values being decomposed.
- `period=12`: there are 12 monthly positions in the yearly seasonal cycle. Do **not** confuse this with the number of years in the dataset.
- `robust=True`: asks STL to reduce the influence of large unusual observations while estimating its smooth components. This is not a promise that unusual observations disappear.
- `.fit()`: calculates the components. Creating `STL(...)` alone does not estimate them.

**The output components:**

| Output | What to say aloud |
|---|---|
| `trend` | Slow-moving estimated level/direction, expressed in thousands of people |
| `seasonal` | Monthly calendar-related difference relative to the trend; centered near zero on average |
| `resid` | Remainder after subtracting estimated trend and seasonal effects |

**Technical note:** Particular smoothing parameters, the length of the observation sample, and `robust=True` can produce slightly different curves from an image prepared using other software/settings. The same original observations are used here.

In [ ]:
# Create an STL decomposition configured for a yearly cycle in monthly data.
stl_model = STL(employment['y'], period=12, robust=True)

# Fit the smoothing procedure; this calculates the estimated component series.
stl_result = stl_model.fit()

# Build a clear table with dates, the actual employment and its three components.
dcmp = pd.DataFrame({
    # Preserve the original monthly date for every component value.
    'ds': employment['ds'],
    # Preserve observed Retail Trade employment in thousands of persons.
    'observed': employment['y'],
    # Store the slowly changing estimated trend-cycle.
    'trend': stl_result.trend,
    # Store each month's seasonal difference from the trend.
    'seasonal': stl_result.seasonal,
    # Store what remains after trend and seasonal effects.
    'remainder': stl_result.resid,
})

# Show actual months so students can connect each component to its observed value.
display(dcmp.head(8))

# Reconstruct the observed values from the estimated additive components.
reconstructed = dcmp['trend'] + dcmp['seasonal'] + dcmp['remainder']

# Verify numerically that the decomposition adds back to the original series.
assert np.allclose(reconstructed, dcmp['observed'], rtol=1e-9, atol=1e-6)

# Print a confirmation rather than asking students to trust an arbitrary graph.
print('STL check passed: observed = trend + seasonal + remainder for this fitted result.')

### 10.1 · How to read a four-component decomposition graph

Look at every panel **in order**:

1. **Observed:** the original measurement that management wants to understand.
2. **Trend:** the broad pattern after shorter-term calendar fluctuations have been smoothed.
3. **Seasonal:** recurrent within-year behavior; above zero means above the estimated trend, below zero means below.
4. **Remainder:** an observation's extra difference after the first two estimated components have been accounted for.

**Do not compare panel heights carelessly:** The original series is around tens of thousands (thousands of persons), while seasonal/remainder variations use a much narrower numeric range. Always read each panel's own y-axis.

In [ ]:
# Create four separate vertical panels sharing the same calendar axis.
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

# Plot actual monthly employment in the first panel.
axes[0].plot(dcmp['ds'], dcmp['observed'])

# Label the first panel as the original data.
axes[0].set_ylabel('Observed')

# Plot the smoother estimated trend in the second panel.
axes[1].plot(dcmp['ds'], dcmp['trend'])

# Label the second panel as a trend, still in thousands of persons.
axes[1].set_ylabel('Trend')

# Plot the estimated monthly seasonal effect in the third panel.
axes[2].plot(dcmp['ds'], dcmp['seasonal'])

# Show a zero reference: positive and negative seasonal differences have meaning.
axes[2].axhline(0, linewidth=0.8, linestyle='--')

# Label the third panel with its component name.
axes[2].set_ylabel('Seasonal')

# Plot the leftover deviation in the bottom panel.
axes[3].plot(dcmp['ds'], dcmp['remainder'])

# Show the zero reference for unusually positive or negative residual movement.
axes[3].axhline(0, linewidth=0.8, linestyle='--')

# Label the last component as the remainder.
axes[3].set_ylabel('Remainder')

# Add the time-axis label once because all four panels share the same dates.
axes[3].set_xlabel('Month')

# Explain both method and business series in the overall chart title.
fig.suptitle('STL decomposition — US Retail Trade employment (thousands)')

# Preserve space for the overall title and keep labels apart.
plt.tight_layout(rect=(0, 0, 1, 0.97))

# Display all four components in order.
plt.show()

## 11 · Observed versus trend, then seasonal adjustment

The *trend* is not the same as the *seasonally adjusted series*:

- `trend` is smoothed; it deliberately removes much of the short-term movement.
- `seasonally_adjusted = observed − seasonal` for an additive decomposition; it **still includes the remainder**. It should generally wiggle more than the smoothed trend.

**Managerial use:** We may adjust employment for repeated calendar effects to compare neighboring months more fairly. Seasonal adjustment is not a future forecast and does not eliminate all economic shocks or changes in the data.

In [ ]:
# Calculate seasonally adjusted employment from the original and STL seasonal estimates.
dcmp['seasonally_adjusted'] = dcmp['observed'] - dcmp['seasonal']

# Verify seasonally adjusted series equals trend plus remainder, up to rounding.
assert np.allclose(dcmp['seasonally_adjusted'], dcmp['trend'] + dcmp['remainder'])

# Start a graph comparing the observed series with just the estimated trend.
plt.figure()

# Draw the actual observed employment line.
plt.plot(dcmp['ds'], dcmp['observed'], label='Observed')

# Draw the smoothed trend-cycle on the same employment scale.
plt.plot(dcmp['ds'], dcmp['trend'], label='STL trend')

# Add a descriptive title and units.
plt.title('US Retail Trade — original data versus STL trend')
plt.xlabel('Month')
plt.ylabel('Employed persons (thousands)')

# Show the legend so students know what each line represents.
plt.legend()

# Fit the labels into the figure and display it.
plt.tight_layout()
plt.show()

# Start a separate figure for seasonal adjustment, which is NOT the trend alone.
plt.figure()

# Draw the unchanged observed line again as a reference.
plt.plot(dcmp['ds'], dcmp['observed'], label='Observed')

# Draw observed minus estimated seasonal contribution.
plt.plot(dcmp['ds'], dcmp['seasonally_adjusted'], label='Seasonally adjusted')

# State precisely which two versions are compared.
plt.title('US Retail Trade — before and after seasonal adjustment')
plt.xlabel('Month')
plt.ylabel('Employed persons (thousands)')

# Make the line identities obvious to students.
plt.legend()

# Display the resulting graph.
plt.tight_layout()
plt.show()

## 12 · Seasonal subseries — do the same months behave consistently?

**Question:** Is December's seasonal employment contribution similar over time? What happens in September? STL allows the seasonal shape to move gradually instead of assuming every year's January must have exactly the same effect.

**How this graph works:** Each small panel contains one month across all available years. For example, the January panel compares **January with January**, not January with February. A line that slopes or changes within the January panel indicates that the estimated January seasonal effect has changed over the sample.

**Important interpretation:** We are plotting the **STL seasonal component**, not raw employment. A higher December seasonal contribution means that December is farther *above its own trend estimate*, not necessarily that its raw employment is highest in the entire year.

In [ ]:
# Copy the original date and the fitted STL seasonal component only.
seasonal_sub = dcmp[['ds', 'seasonal']].copy()

# Extract the integer month number, with 1 for January through 12 for December.
seasonal_sub['month_number'] = seasonal_sub['ds'].dt.month

# Extract a short month label, such as Jan or Feb, for graph titles.
seasonal_sub['month_name'] = seasonal_sub['ds'].dt.strftime('%b')

# Extract the year to put each month's seasonal contribution on an annual axis.
seasonal_sub['year'] = seasonal_sub['ds'].dt.year

# Create one small panel per month (3 rows × 4 columns = 12 months).
fig, axes = plt.subplots(3, 4, figsize=(15, 9), sharex=True, sharey=True)

# Loop through all calendar months in chronological order.
for month_number in range(1, 13):
    # Select only January rows, then February rows, and so on for this iteration.
    this_month = seasonal_sub.loc[seasonal_sub['month_number'] == month_number]

    # Find the matching position in the 3×4 array of panels.
    ax = axes.flat[month_number - 1]

    # Plot seasonal effect by year for this one calendar month.
    ax.plot(this_month['year'], this_month['seasonal'])

    # Label each subplot with its month rather than a confusing number 1–12.
    ax.set_title(this_month['month_name'].iloc[0])

    # Draw zero to distinguish above-trend from below-trend seasonal effects.
    ax.axhline(0, linewidth=0.6, linestyle='--')

# Give one overall title identifying what the 12 panels mean.
fig.suptitle('US Retail Trade — STL seasonal effect by month across years')

# Add a compact global time label beneath the whole grid.
fig.supxlabel('Year')

# Add a global vertical-axis label and its units.
fig.supylabel('Seasonal employment effect (thousands)')

# Allow enough room for the overall title, axis labels and 12 small panels.
plt.tight_layout(rect=(0.03, 0.04, 1, 0.96))

# Show the seasonal subseries figure.
plt.show()

---
# Part D · Moving averages: seeing the trend-cycle

A **moving average (MA)** replaces a point by the average of nearby observations. It can make trend-cycle movement clearer by reducing short-term variation. *MA order* `m` is the number of observations used in each average; it is **not** the number of years automatically.

**Trailing versus centered:**

- **Trailing 3-MA** at month `t`: `(y[t−2] + y[t−1] + y[t]) / 3`. It uses current and earlier observations and is potentially available at the latest date.
- **Centered 3-MA** at month `t`: `(y[t−1] + y[t] + y[t+1]) / 3`. It uses one **future** observation relative to the central month, so it helps describe historical trend but cannot be evaluated at the latest observed month without another observation arriving.
- **Centered 5-MA** uses two months (or years) on each side. A larger order usually smooths more but removes more endpoints.

**Dataset for this section:** `global_economy.csv`, `unique_id == 'Australia'`, annual `Exports`. The `Exports` column is recorded as a **percent of GDP**. Thus `15` means 15% of GDP, not 15 dollars or 15 exported goods.

## 13 · Australian exports: one 5-period centered MA

**Why use five?** In this annual series, a centered 5-MA uses two years before, the target year, and two years after; it can show a smoother historical path. To forecast future exports we would need a separate method—the centered 5-MA is a *description* of the past, not a forecasting model on its own.

In [ ]:
# Select the Australia economy rows already prepared and keep year and Exports.
aus_exports = australia[['ds', 'Exports']].copy()

# Make the date order explicit before using a moving window.
aus_exports = aus_exports.sort_values('ds').reset_index(drop=True)

# Calculate a centered five-year moving average with two years on each side.
aus_exports['ma_5'] = aus_exports['Exports'].rolling(window=5, center=True).mean()

# Show actual export shares and smoothed shares in the same unit: percentage points of GDP.
display(aus_exports[['ds', 'Exports', 'ma_5']].head(8))

# Start a graph containing both annual exports and their historical smoother.
plt.figure()

# Plot the original series to preserve all observed ups and downs.
plt.plot(aus_exports['ds'], aus_exports['Exports'], label='Observed Exports')

# Plot the centered 5-year average over the same years.
plt.plot(aus_exports['ds'], aus_exports['ma_5'], label='Centered 5-MA')

# State the geographic area and what the moving average smooths.
plt.title('Australian exports — annual observations and centered 5-MA')

# Specify annual date units.
plt.xlabel('Year')

# Use the correct exports unit from the source file.
plt.ylabel('Exports (% of GDP)')

# Display a legend distinguishing raw and smoothed curves.
plt.legend()

# Keep chart labels visible and show the graph.
plt.tight_layout()
plt.show()

## 14 · Compare 3-, 5-, 7-, 9-, 11-, 13- and 15-period MAs

The *same original annual series* is smoothed repeatedly with a different odd order. As the order increases, the estimate usually becomes flatter and responds less to rapid swings. This is a **trade-off**, not evidence that the largest order must be selected.

**What `.rolling(window=m, center=True).mean()` means:** `window=m` chooses the number of observations; `center=True` assigns their mean to the center of the window; `.mean()` calculates the arithmetic average. For centered odd orders, pandas returns missing values near both ends where enough neighbors do not exist.

In [ ]:
# Specify the exact odd orders taught in the lecture.
orders = [3, 5, 7, 9, 11, 13, 15]

# Compute each centered average from the ORIGINAL exports column, not a previous MA.
for order in orders:
    # Construct a separate column name such as ma_3 or ma_15.
    column_name = f'ma_{order}'

    # Use this number of consecutive annual observations, centered on each year.
    aus_exports[column_name] = aus_exports['Exports'].rolling(window=order, center=True).mean()

# Start a chart combining a few orders; all seven can become visually cluttered.
plt.figure()

# Keep the original series visible as the benchmark.
plt.plot(aus_exports['ds'], aus_exports['Exports'], label='Original', alpha=0.65)

# Add a relatively responsive, short centered average.
plt.plot(aus_exports['ds'], aus_exports['ma_3'], label='3-MA')

# Add a middle-length centered average.
plt.plot(aus_exports['ds'], aus_exports['ma_7'], label='7-MA')

# Add the smoothest of the selected historical averages.
plt.plot(aus_exports['ds'], aus_exports['ma_15'], label='15-MA')

# Describe why we have more than one curve.
plt.title('Australian exports — smoothing changes with MA order')

# Use annual time units and original exports units.
plt.xlabel('Year')
plt.ylabel('Exports (% of GDP)')

# Label the individual lines rather than asking students to guess by appearance.
plt.legend()

# Draw the comparison chart.
plt.tight_layout()
plt.show()

## 15 · Endpoint loss — why do the smoothed lines stop early?

For a centered odd-order moving average, there must be `(m − 1) / 2` observations **before and after** the target year. Thus a centered 3-MA has one unavailable value at each end; a centered 15-MA has seven at each end, assuming the original series has no missing observations.

This is a real disadvantage if someone asks, “What is the smoothed trend **today**?” Future observations required for centering are not yet available. Do not fill the endpoints with zeros—they are **undefined**, not observed zeros.

In [ ]:
# Print counts of missing values in each centered moving average column.
for order in orders:
    # Count NaN values introduced because the centered window cannot fit at an endpoint.
    missing_count = int(aus_exports[f'ma_{order}'].isna().sum())

    # Calculate the theoretical number of lost endpoints in a complete annual series.
    expected_missing = order - 1

    # Display a short comparison for every order.
    print(f'{order}-MA: {missing_count} missing endpoints; expected {expected_missing}')

    # Verify the observed result matches the theoretical endpoint loss in this dataset.
    assert missing_count == expected_missing, f'Check missing values in {order}-MA.'

# Demonstrate the first non-missing 5-MA equals the average of the first five real years.
manual_first_ma5 = aus_exports['Exports'].iloc[0:5].mean()

# Read pandas' first defined centered 5-MA, which is aligned at the THIRD year.
computed_first_ma5 = aus_exports['ma_5'].iloc[2]

# Check students get the same answer from manual arithmetic and Python's rolling method.
assert np.isclose(manual_first_ma5, computed_first_ma5)

# Show actual numbers so the rolling-average concept is not only a picture.
print('Manual average of first five exports values:', round(manual_first_ma5, 5))
print('Centered 5-MA at the third year:', round(computed_first_ma5, 5))

## 16 · Even-order centered averages: why 2×4 for quarterly data?

**Dataset:** `aus_production.csv`, with a real quarterly date `ds` and quarterly `Beer` production. We keep 1992 onward to focus on the lecture's modern quarterly example; this is **not simulated**.

Quarterly annual seasonality has `m=4` positions: Q1, Q2, Q3, Q4. But the center of **four consecutive observations lies between two quarters**, not exactly on one. A standard trend-cycle estimator is a **2×4 moving average**:

1. Calculate all consecutive **4-quarter averages**.
2. Average every two neighboring 4-quarter averages.
3. Align the result to the actual central quarter.

Its weights on five consecutive quarterly observations are `(1, 2, 2, 2, 1) / 8`.

**Why a centered 4-MA alone is not enough:** It sits conceptually half a quarter away from the center date. Our second average corrects that alignment. The 2×4 average smooths an entire four-quarter seasonal cycle when seasonal effects are stable; it does not necessarily eliminate all irregular variation.

In [ ]:
# Keep Beer production from January 1992 onward and ignore other production columns.
beer = aus_production.loc[aus_production['ds'] >= '1992-01-01', ['ds', 'Beer']].copy()

# Sort the quarterly observations and use simple row numbers for moving-window calculations.
beer = beer.sort_values('ds').reset_index(drop=True)

# Verify Beer production is observed for the entire selected sample.
assert beer['Beer'].notna().all(), 'The Beer series has missing observations.'

# Verify quarter-start dates advance as Q1, Q2, Q3, Q4 without missing quarters.
expected_quarters = pd.date_range(beer['ds'].min(), beer['ds'].max(), freq='QS-JAN')

# Stop if a period is absent, because row-based rolling would then not mean four quarters.
assert beer['ds'].tolist() == list(expected_quarters), 'Missing quarterly periods in Beer data.'

# Calculate successive centered 4-observation means (each is centered between quarters).
beer['intermediate_4ma'] = beer['Beer'].rolling(window=4, center=True).mean()

# Average neighboring 4-MAs and shift ONE row to align at the correct central quarter.
beer['ma_2x4'] = beer['intermediate_4ma'].rolling(window=2).mean().shift(-1)

# Show genuine observed values, intermediate 4-MA and final 2×4-MA.
display(beer[['ds', 'Beer', 'intermediate_4ma', 'ma_2x4']].head(9))

# Calculate the weighted five-point formula BY HAND for the first defined center.
manual_2x4 = (beer['Beer'].iloc[0] + 2 * beer['Beer'].iloc[1]
              + 2 * beer['Beer'].iloc[2] + 2 * beer['Beer'].iloc[3]
              + beer['Beer'].iloc[4]) / 8

# Compare the manual weighted sum with the rolling calculation at the third quarter.
assert np.isclose(manual_2x4, beer['ma_2x4'].iloc[2])

# Show the two identical outputs as a step-by-step mathematical check.
print('Manual 2×4 average for center at row 2:', manual_2x4)
print('Pandas 2×4 average at row 2:', beer['ma_2x4'].iloc[2])

# Begin a graph of original quarterly Beer production and its moving-average estimate.
plt.figure()

# Plot the unsmoothed quarter-by-quarter production.
plt.plot(beer['ds'], beer['Beer'], label='Observed Beer production')

# Overlay the centered 2×4-MA estimate.
plt.plot(beer['ds'], beer['ma_2x4'], label='Centered 2×4-MA')

# Label the quarterly production example clearly.
plt.title('Australian Beer production — original and 2×4-MA')

# Use the correct quarterly time unit.
plt.xlabel('Quarter')

# Use the dataset's original production units without inventing a new physical unit.
plt.ylabel('Beer production (source units)')

# Show the line labels.
plt.legend()

# Display the graph.
plt.tight_layout()
plt.show()

## 17 · Monthly seasonality: a 2×12 moving average

The exact same even-order issue appears when monthly data have **annual seasonality**: `m=12`. A centered 12-month window lies between two calendar months. We therefore compute **2×12-MA** to put the trend-cycle estimate back on an actual month.

We reuse `employment`, the **original US Retail Trade** monthly series from 1990. This is a moving-average trend estimate, not STL. The two techniques may produce different trend lines because they estimate components differently.

In [ ]:
# Copy the monthly Retail Trade table so we do not alter the original employment object.
employment_ma = employment[['ds', 'y']].copy()

# Compute centered 12-month means; these initially lie between neighboring months.
employment_ma['intermediate_12ma'] = employment_ma['y'].rolling(window=12, center=True).mean()

# Average adjacent 12-MAs and shift the output by one row for calendar-month alignment.
employment_ma['ma_2x12'] = employment_ma['intermediate_12ma'].rolling(window=2).mean().shift(-1)

# Assert that the first and last six months cannot have a complete centered 2×12 window.
assert employment_ma['ma_2x12'].isna().sum() == 12

# Display the first valid aligned month and its estimated trend-cycle value.
display(employment_ma[['ds', 'y', 'ma_2x12']].iloc[4:11])

# Create an observed-versus-2×12 trend graph.
plt.figure()

# Plot original monthly employment, preserving the raw calendar variation.
plt.plot(employment_ma['ds'], employment_ma['y'], label='Observed employment')

# Plot the smoother, season-length centered moving average.
plt.plot(employment_ma['ds'], employment_ma['ma_2x12'], label='2×12-MA trend')

# Give a title specifying series, estimator and seasonal frequency.
plt.title('US Retail Trade — trend-cycle from centered 2×12-MA')

# Label month and employment in thousands of people.
plt.xlabel('Month')
plt.ylabel('Employed persons (thousands)')

# Make the two lines identifiable.
plt.legend()

# Display the finished figure.
plt.tight_layout()
plt.show()

---
# Part E · Classical decomposition

Classical decomposition is a simpler, historical approach built from centered moving averages and one seasonal profile repeated every year. The two common forms are:

**Additive:** `observed = trend + seasonal + remainder`.

**Multiplicative:** `observed = trend × seasonal × remainder`.

**How classical decomposition is constructed:**

1. Estimate trend with an `m-MA` for odd seasonal periods or `2×m-MA` for even periods.
2. Detrend: `observed − trend` (additive) **or** `observed / trend` (multiplicative).
3. Average detrended values for the same season across years; normalize so additive indices sum to 0 or multiplicative indices average 1.
4. Repeat those seasonal indices across the dates.
5. Calculate the remainder after trend and seasonality are removed.

**Major limitation:** The classical seasonal profile repeats without gradually adapting. It can struggle if the seasonal pattern changes or there are strong outliers; its centered trend also lacks endpoint estimates.

**Dataset:** The exact same positive Retail Trade employment values; using one consistent dataset makes the two models comparable. We do not manufacture data for a prettier graph.

## 18 · Classical additive decomposition of US Retail Trade

**Interpret its seasonal values in people:** The dataset's `y` is in thousands, so an additive seasonal value `+200` represents about **200,000 more employed persons than the trend** for that calendar month—not a 200% increase.

`seasonal_decompose` estimates the trend using a centered moving average and estimates a repeating monthly seasonal profile. Because the centered trend needs observations outside the endpoints, the first and last months of its trend/remainder will be `NaN`. `extrapolate_trend` is deliberately **not** used; we want students to see the missing endpoints rather than filling them artificially.

In [ ]:
# Create a monthly indexed Series from the correctly selected Retail Trade employment.
retail_series = employment.set_index('ds')['y']

# Run CLASSICAL additive decomposition for the twelve months of a yearly cycle.
classical_add = seasonal_decompose(retail_series, model='additive', period=12)

# Store components side by side so each date can be inspected numerically.
add_df = pd.DataFrame({
    # Keep the original measured employment.
    'observed': classical_add.observed,
    # Store centered-moving-average trend estimates (undefined at some endpoints).
    'trend': classical_add.trend,
    # Store the repeating monthly additive seasonal amount.
    'seasonal': classical_add.seasonal,
    # Store the remaining additive difference after trend and seasonality.
    'remainder': classical_add.resid,
})

# Print an actual middle-of-sample segment where the trend is defined.
display(add_df.iloc[12:18])

# Keep only rows with all three estimated components available for validation.
add_valid = add_df.dropna()

# Reconstruct observed employment as trend plus seasonal plus remainder.
add_reconstruction = add_valid['trend'] + add_valid['seasonal'] + add_valid['remainder']

# Check that the decomposition calculations reproduce the original values.
assert np.allclose(add_reconstruction, add_valid['observed'])

# Show the actual number of undefined trend/remainder endpoint observations.
print('Classical additive missing trend values:', add_df['trend'].isna().sum())

# Confirm the arithmetic identity holds for all valid dates.
print('Additive reconstruction check passed.')

In [ ]:
# Set up four stacked panels to inspect the classical additive components.
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

# Draw the original monthly employment in the first panel.
axes[0].plot(add_df.index, add_df['observed'])

# Label the original data panel.
axes[0].set_ylabel('Observed')

# Draw classical centered-moving-average trend in panel two.
axes[1].plot(add_df.index, add_df['trend'])

# Label the historical trend estimator.
axes[1].set_ylabel('Trend')

# Draw the repeating additive seasonal amount in panel three.
axes[2].plot(add_df.index, add_df['seasonal'])

# Add zero to distinguish a seasonal boost from a seasonal reduction.
axes[2].axhline(0, linewidth=0.8, linestyle='--')

# Label the additive seasonal component.
axes[2].set_ylabel('Seasonal')

# Draw the leftover additive remainder in panel four.
axes[3].plot(add_df.index, add_df['remainder'])

# Add a zero baseline for the irregular movement.
axes[3].axhline(0, linewidth=0.8, linestyle='--')

# Label the remainder as distinct from the original observations.
axes[3].set_ylabel('Remainder')

# Name the shared date axis.
axes[3].set_xlabel('Month')

# Tell students which estimation approach made these components.
fig.suptitle('Classical additive decomposition — US Retail Trade employment')

# Fit labels and title without overlap.
plt.tight_layout(rect=(0, 0, 1, 0.97))

# Display the four-panel figure.
plt.show()

## 19 · Classical multiplicative decomposition

**Why a different version?** If repeated seasonal differences get proportionally larger as employment changes, seasonal factors may be more interpretable than fixed additions. For example, `1.02` means **2% above estimated trend**, not two persons or two percent of an employment count.

This method requires **strictly positive observations**, because its detrending step divides by trend and its remainder is a ratio. The same observed employment series must remain positive throughout; we verify that explicitly.

**Keep two uses separate:** `STL` earlier is additive. `seasonal_decompose(..., model='multiplicative')` below is a *different method* rather than STL magically changing form.

In [ ]:
# Confirm that no employment observation is zero or negative before multiplication/ratios.
assert (retail_series > 0).all(), 'Multiplicative decomposition needs strictly positive values.'

# Estimate the classical trend, multiplicative seasonality and remainder.
classical_mul = seasonal_decompose(retail_series, model='multiplicative', period=12)

# Put multiplicative component values in an easy-to-check dated table.
mul_df = pd.DataFrame({
    # Original employment is unchanged from the additive example.
    'observed': classical_mul.observed,
    # The trend is a smoothed estimate of employment level.
    'trend': classical_mul.trend,
    # Multiplicative seasonality is a RATIO centered around one, not zero.
    'seasonal_factor': classical_mul.seasonal,
    # Multiplicative remainder is also a ratio centered around one.
    'remainder_factor': classical_mul.resid,
})

# Display real middle-of-sample values and factor sizes.
display(mul_df.iloc[12:18])

# Drop undefined endpoint trends before checking multiplicative arithmetic.
mul_valid = mul_df.dropna()

# Multiply three components to reconstruct each original employment value.
mul_reconstruction = mul_valid['trend'] * mul_valid['seasonal_factor'] * mul_valid['remainder_factor']

# Verify reconstruction, allowing tiny floating-point calculation differences.
assert np.allclose(mul_reconstruction, mul_valid['observed'])

# Print a short verification message after the ratio arithmetic matches the source.
print('Multiplicative reconstruction check passed.')

In [ ]:
# Create four vertically stacked component panels for multiplicative results.
fig, axes = plt.subplots(4, 1, figsize=(12, 10), sharex=True)

# Draw original monthly employment on its original thousand-person scale.
axes[0].plot(mul_df.index, mul_df['observed'])

# Label the unchanged original series.
axes[0].set_ylabel('Observed')

# Plot the centered-moving-average trend.
axes[1].plot(mul_df.index, mul_df['trend'])

# Label the trend panel.
axes[1].set_ylabel('Trend')

# Plot relative seasonal factors around one.
axes[2].plot(mul_df.index, mul_df['seasonal_factor'])

# Show the neutral multiplicative factor 1, meaning no seasonal change.
axes[2].axhline(1, linewidth=0.8, linestyle='--')

# Describe seasonality as a ratio rather than as employment in thousands.
axes[2].set_ylabel('Seasonal factor')

# Plot the remaining ratio after estimated trend and seasonality.
axes[3].plot(mul_df.index, mul_df['remainder_factor'])

# Show the neutral multiplicative remainder factor.
axes[3].axhline(1, linewidth=0.8, linestyle='--')

# Label this panel as a factor, not an additive difference.
axes[3].set_ylabel('Remainder factor')

# Give the common timeline its unit.
axes[3].set_xlabel('Month')

# Name the method precisely so it cannot be confused with additive STL.
fig.suptitle('Classical multiplicative decomposition — US Retail Trade employment')

# Prevent title and subplot labels from overlapping.
plt.tight_layout(rect=(0, 0, 1, 0.97))

# Show the full comparison.
plt.show()

## 20 · Monthly seasonal indices and seasonal adjustment — verify meaning in numbers

A seasonal index summarizes the pattern associated with one part of the calendar. To estimate “the December effect,” we look at December observations across the available years rather than averaging every month together. In **classical additive decomposition**, the twelve estimated indices should sum to approximately **zero**; in classical multiplicative decomposition, their average should be approximately **one**.

**Seasonal adjustment formulas:**

- Additive: `observed − seasonal_amount`.
- Multiplicative: `observed / seasonal_factor`.

**Be careful:** Seasonal adjustment is not identical to taking a moving average. The seasonally adjusted series still contains the remainder. The purpose is easier comparison after removing the estimated repeating calendar contribution.

In [ ]:
# Take month number from each dated row in the additive decomposition result.
month_num = add_df.index.month

# Average the repeating additive seasonal values for each month of the year.
add_indices = add_df.groupby(month_num)['seasonal'].mean()

# Average the repeating multiplicative seasonal factors for each month of the year.
mul_indices = mul_df.groupby(mul_df.index.month)['seasonal_factor'].mean()

# Assemble a side-by-side month, additive index and multiplicative factor table.
seasonal_index_table = pd.DataFrame({
    # Give the month a readable abbreviation derived from the original dates.
    'Month': pd.date_range('2000-01-01', periods=12, freq='MS').strftime('%b'),
    # Take month 1 through month 12 in the correct calendar order.
    'Additive_index_thousands': add_indices.to_numpy(),
    # Keep multiplicative values around one so a percentage interpretation is possible.
    'Multiplicative_factor': mul_indices.to_numpy(),
})

# Convert multiplicative ratios into the percent above/below estimated trend.
seasonal_index_table['Percent_vs_trend'] = (seasonal_index_table['Multiplicative_factor'] - 1) * 100

# Display the actual twelve index values calculated from the book dataset.
display(seasonal_index_table)

# Check the classical additive normalization: total seasonal effect across a year is zero.
assert np.isclose(seasonal_index_table['Additive_index_thousands'].sum(), 0, atol=1e-6)

# Check the multiplicative normalization: the mean factor is one.
assert np.isclose(seasonal_index_table['Multiplicative_factor'].mean(), 1, atol=1e-6)

# Compute the additive seasonally adjusted series by removing the monthly amount.
add_df['seasonally_adjusted'] = add_df['observed'] - add_df['seasonal']

# Compute the multiplicative seasonally adjusted series by dividing out the month factor.
mul_df['seasonally_adjusted'] = mul_df['observed'] / mul_df['seasonal_factor']

# Show a short slice to distinguish original and two adjustment scales.
display(pd.DataFrame({
    # Include unchanged original employment.
    'Original': add_df['observed'],
    # Include observed minus classical additive seasonal contribution.
    'Additive_adjusted': add_df['seasonally_adjusted'],
    # Include observed divided by classical multiplicative seasonal factor.
    'Multiplicative_adjusted': mul_df['seasonally_adjusted'],
}).iloc[12:18])

# Confirm seasonal indices are correctly normalized.
print('Normalization checks passed: additive indices sum to 0; multiplicative factors average 1.')

## 21 · What should you remember about the method choice?

These methods do not solve exactly the same problem:

| Tool | Input/setting | Output and limitation |
|---|---|---|
| Time plot | date + observed values | First diagnosis of trend, seasonality, unusual points |
| Log transform | strictly positive observations | Changes the scale; does not automatically remove seasonality |
| Centered `m`-MA | odd `m` | Smoothed historical trend estimate, loses endpoints |
| Centered `2×m`-MA | even `m`, such as 4 or 12 | Season-length centered historical trend, loses endpoints |
| STL (`period=12`) | monthly series with annual seasonality | Flexible additive trend/seasonal/remainder estimates |
| Classical additive | fairly stable absolute seasonal effects | One repeating seasonal *amount* each year |
| Classical multiplicative | strictly positive data, proportional seasonal effects | One repeating seasonal *factor* each year |

**Avoid three common errors:** (1) interpreting a 12-month moving average as a *forecast* of future values; (2) treating a seasonal factor `1.03` as an additive increase of 1.03 thousand people; (3) calling data “seasonally adjusted” when only the smoothed trend is plotted.

---
# Part F · “Remember these for later”: your independent-notebook reference

The following snippets deliberately repeat important code. They are **templates**: reuse them when starting a new assignment, and adapt the **file name, column name, target industry, time frequency and seasonal period** to the dataset you actually have. They all execute with the same four uploaded lecture files.

**Minimal workflow to remember:** import packages → upload → read CSV → inspect → convert/sort dates → filter → graph the original → choose a valid method → interpret the output → verify units and arithmetic.

## 22 · Template: begin a new notebook, locate a CSV, check its columns

If you start from a blank notebook, run the package imports, run the ZIP/CSV upload section, then use this pattern. Uploading a file and reading a file are **different actions**: `pd.read_csv()` needs a file present at the path you specify.

In [ ]:
# Import pandas whenever a NEW notebook needs tables and CSV reading.
import pandas as pd

# Import plotting functions whenever a NEW notebook needs graphs.
import matplotlib.pyplot as plt

# Import Path to make safe reusable file paths.
from pathlib import Path

# Set the same local folder used by this lecture's upload cell.
student_data_folder = Path('Lecture2_Data')

# Read ONE original book CSV as a practical template for future assignments.
example_df = pd.read_csv(student_data_folder / 'global_economy.csv')

# Show the first five rows to learn what one observation represents.
display(example_df.head())

# Print exact column names: Python is case-sensitive (GDP is not gdp).
print('Columns:', example_df.columns.tolist())

# Print number of observations and columns so we know our sample's size.
print('Shape:', example_df.shape)

# Inspect missing cells instead of assuming every observation is complete.
print('Missing values:', example_df.isna().sum().to_dict())

## 23 · Template: convert monthly dates, filter a series and draw a time plot

For monthly data, use `parse_dates=['ds']` when loading a file (or `pd.to_datetime` afterward). For annual economy data, `ds` is just a year; do not needlessly pretend it is a month. **Always sort** before `.rolling()`. A time plot must use date on the horizontal axis, not a random row index when dates are available.

In [ ]:
# Open the original monthly employment dataset and parse its dates immediately.
new_employment = pd.read_csv(student_data_folder / 'us_employment.csv', parse_dates=['ds'])

# Choose one industry so different series do not become mixed in the same time line.
new_retail = new_employment.loc[new_employment['unique_id'] == 'Retail Trade'].copy()

# Sort by date and drop old row numbers for easy reading.
new_retail = new_retail.sort_values('ds').reset_index(drop=True)

# Plot the selected industry's actual dated observations.
plt.figure()
plt.plot(new_retail['ds'], new_retail['y'])

# Explain both axes and identify what is being shown.
plt.title('Template: US Retail Trade monthly employment')
plt.xlabel('Month')
plt.ylabel('Employed persons (thousands)')

# Show the graph after arranging labels neatly.
plt.tight_layout()
plt.show()

## 24 · Template: trailing MA, centered MA, even-order MA and STL

This last template puts four often-confused operations next to each other:

- **Trailing 3-MA**: current plus two earlier months. The first two values are missing.
- **Centered 3-MA**: previous, current and next month. The first and last values are missing.
- **Centered 2×12-MA**: historical annual-season-length trend, loses six at each end in a complete monthly series.
- **STL**: a fitted decomposition, not just a rolling average.

The code checks alignment and confirms that the fitted STL components reconstruct the original measurements.

In [ ]:
# Keep only a new copy of the monthly employment dates and measured values.
practice = employment[['ds', 'y']].copy()

# Trailing 3-MA: average the current month and two earlier months.
practice['trailing_3ma'] = practice['y'].rolling(window=3).mean()

# Centered 3-MA: average previous, current and next month.
practice['centered_3ma'] = practice['y'].rolling(window=3, center=True).mean()

# Begin even-order centering with the interim centered 12-month averages.
interim_12ma = practice['y'].rolling(window=12, center=True).mean()

# Average neighboring interim values and align them to the actual calendar month.
practice['centered_2x12ma'] = interim_12ma.rolling(window=2).mean().shift(-1)

# Fit STL for the correct twelve-month yearly cycle in this monthly series.
practice_stl = STL(practice['y'], period=12, robust=True).fit()

# Store the trend from this new, independently fitted STL model.
practice['stl_trend'] = practice_stl.trend

# Store the seasonal contribution from the fitted STL model.
practice['stl_seasonal'] = practice_stl.seasonal

# Store the remainder not described by trend/seasonality.
practice['stl_remainder'] = practice_stl.resid

# Build a seasonally adjusted monthly series by subtracting the estimated seasonal amount.
practice['seasonally_adjusted'] = practice['y'] - practice['stl_seasonal']

# Display a short sample that contains both moving averages and decomposition results.
display(practice.iloc[7:13].round(3))

# Verify the additive STL arithmetic, preventing a silent row-alignment mistake.
assert np.allclose(practice['y'], practice['stl_trend'] + practice['stl_seasonal'] + practice['stl_remainder'])

# Tell students they can reuse the calculation structure in later assignments.
print('Blank-notebook practice completed: moving averages and STL reconstruction verified.')

## 25 · Quick self-check before submitting an assignment

Try to answer each question by checking the corresponding graph or printed numbers in **this notebook**:

1. In the calendar adjustment, what unit did you obtain after dividing monthly food turnover by month length? Why is this not automatically “sales per trading day”?
2. In the per-capita example, how does `GDP / Population` change the business question? Does this calculation remove inflation?
3. Why do nominal and CPI-adjusted annual print-media turnover match in 2010?
4. What restriction must hold before you calculate `np.log(y)` or `1/y`?
5. For the employment data, why is the STL seasonal period 12? Is `y` measured in persons or thousands of persons?
6. Explain why seasonally adjusted employment is not the same line as the STL trend.
7. What happens to the beginning and end of a centered 15-MA, and why?
8. Why do quarterly observations with yearly seasonality use 2×4-MA rather than an unadjusted centered 4-MA?
9. If a multiplicative seasonal factor is 1.04, what is the interpretation? Would a classical additive seasonal value of +200 have the same units?
10. Can any of these **historical** decomposition graphs by themselves guarantee an accurate forecast of the next observation?

### Key ideas to remember

**Source first, graph second, interpretation third.** Keep the data's geographic coverage, date frequency, measurement unit and transformation visible as you work. If a calculation runs but answers the wrong question, it is still a poor analysis.

**The entire notebook is based only on the four uploaded original book CSVs. No simulated series and no online data downloads are used.**